Now we have user_day_master.csv (330,452 rows, all 5 sources combined), it's time to split it properly by time — so model only learns from "clean" history and gets tested on the period containing the actual threats.

### Load the master table and check its date range

In [5]:
import pandas as pd
master = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/user_day_master.csv")
master['day'] = pd.to_datetime(master['day'])

print("Date range:", master['day'].min(), "to", master['day'].max())
print("Total user-days:", master.shape[0])

Date range: 2010-01-02 00:00:00 to 2011-05-17 00:00:00
Total user-days: 330452


### Check when your malicious scenarios actually happened

In [6]:
answers = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/answers_r42.csv")
answers['start'] = pd.to_datetime(answers['start'])
answers['end'] = pd.to_datetime(answers['end'])

print(answers[['user','start','end']].sort_values('start'))

       user               start                 end
63  CSC0217 2010-06-10 07:54:10 2010-06-11 17:42:48
52  PNL0301 2010-06-14 09:25:31 2010-08-03 16:54:30
48  LCC0819 2010-06-16 11:39:58 2010-08-10 18:31:17
64  GTD0219 2010-06-17 09:06:37 2010-06-18 17:50:42
56  RMW0542 2010-06-21 09:04:43 2010-08-18 17:37:38
..      ...                 ...                 ...
16  KLH0596 2011-02-12 07:11:50 2011-02-12 07:33:24
41  HBO0413 2011-02-14 08:15:24 2011-04-08 15:25:10
35  CQW0652 2011-02-18 09:22:22 2011-04-14 20:58:56
29  WDD0366 2011-02-24 19:49:41 2011-03-03 01:01:02
66  JLM0364 2011-04-28 09:50:07 2011-04-29 20:04:27

[70 rows x 3 columns]


### Select Split Date

In [7]:
earliest_scenario_start = answers['start'].min()
print("Earliest scenario starts at:", earliest_scenario_start)

total_days = (master['day'].max() - master['day'].min()).days
print("Total days in dataset:", total_days)

Earliest scenario starts at: 2010-06-10 07:54:10
Total days in dataset: 500


In [8]:
split_date = pd.Timestamp("2010-06-01")

Here train period can only safely span Jan 2 – ~June 10, 2010 — roughly the first 32% of the timeline.
This sits safely 9 days before the earliest scenario start (June 10), giving a small buffer — useful because some feature engineering later (like rolling 7-day averages) needs a few days of lead-time to compute properly without touching scenario data.

In [9]:
print("Any scenarios starting before split date?", (answers['start'] < split_date).any())

Any scenarios starting before split date? False


In [10]:
train_df = master[master['day'] < split_date].copy()
test_df = master[master['day'] >= split_date].copy()

print("Train shape:", train_df.shape, "-> ~%.0f%% of data" % (100*len(train_df)/len(master)))
print("Test shape:", test_df.shape, "-> ~%.0f%% of data" % (100*len(test_df)/len(master)))

Train shape: (105171, 7) -> ~32% of data
Test shape: (225281, 7) -> ~68% of data


## Train/Test Split — Design Note
The CERT r4.2 dataset's earliest injected scenario begins June 10, 2010, 
just ~5 months into the ~16.5-month dataset. This constrains the training 
period to roughly the first 32% of the timeline (Jan 2 – June 1, 2010) 
rather than a more typical 70-75% split, since no scenario data can be 
included in training without leakage. This is a structural characteristic 
of the r4.2 dataset (70 malicious users active across a wide, overlapping 
timeframe) rather than a modeling choice, and is noted as a limitation: 
the model has a smaller "clean" baseline period to learn normal behavior 
from than would be ideal.

- **Split date:** 2010-06-01
- **Train period:** 2010-01-02 to 2010-06-01 (~32% of data)
- **Test period:** 2010-06-01 to 2011-05-17 (~68% of data, contains all scenarios)

In [11]:
train_df.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/train_user_day.csv", index=False)
test_df.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/test_user_day.csv", index=False)

print("Saved train:", train_df.shape)
print("Saved test:", test_df.shape)

Saved train: (105171, 7)
Saved test: (225281, 7)
